<div style="
  background: linear-gradient(135deg, #667eea, #764ba2, #ff9a9e);
  padding: 32px;
  border-radius: 24px;
  text-align: center;
  color: #ffffff;
  box-shadow: 0 8px 20px rgba(0,0,0,0.15);
  margin-bottom: 20px;
">
  <h1 style="font-size: 42px; margin-bottom: 8px;">🎲 Bayesian Probability Simulation 🔬</h1>
  <h2 style="font-size: 24px; margin-top: 0; color: #f0e6ff;">From Conditional Probability to Naive Bayes Classification</h2>
  <p style="font-size: 18px; line-height: 1.7; color: #f0e6ff;">
    A refactored, simulation-driven exploration of Bayesian inference, Monte Carlo convergence,<br>
    and text classification — with corrected theory and reproducible R code.
  </p>
  <div style="background-color: rgba(255,255,255,0.18); display: inline-block; padding: 10px 18px;
              border-radius: 18px; margin-top: 12px; font-size: 16px; color: #ffffff;">
    R &nbsp;•&nbsp; Bayes' Theorem &nbsp;•&nbsp; Law of Total Probability &nbsp;•&nbsp; Monte Carlo &nbsp;•&nbsp; Naive Bayes
  </div>
</div>

### Notebook Overview

This notebook refactors the original course assignment into a **question-free, explanation-driven** format. Each section presents the **analytical derivation** followed immediately by a **Monte Carlo simulation** that validates it.

| Part | Topic | Core Concept |
|------|-------|--------------|
| 1.1 | Airport Security Scanner | Law of Total Probability & Bayes' theorem, simulation convergence |
| 1.2 | Fraud Detection | Bayes' theorem with one and two conditionally independent features |
| 2.1 | Monty Hall Problem | Conditional probability, switching vs staying |
| 2.2 | Infinite Monkey Theorem | Rare events, geometric / binomial intuition, `1-(1-p)^N` |
| 3 | Spam Email Classification | Bernoulli Naive Bayes from scratch, data leakage avoidance, Laplace smoothing |

**Reproducibility:** `set.seed(42)` (or `123`) is set before every stochastic block. Analytical values are printed alongside empirical estimates.

**Changes vs original:** Fixed deterministic-shuffle simulation (`rep`+`sample` → `rbinom`), wrong `84/113` → `84/133`, binary `any()` 0/1 bug in monkey convergence, data-leakage in spam DTM, Laplace denominator `+V` → `+2`, plus all convergence plots now use replicated runs with confidence bands.

In [ ]:
# Global reproducibility and libraries
set.seed(42)
# Required libraries (install once externally: install.packages(c("tm","SnowballC","e1071","caret")))
library(tm)
library(SnowballC)
library(e1071)
# library(caret)  # not needed — manual stratified sampling used


<div style="
  background: linear-gradient(135deg, #74c69d, #48cae4);
  padding: 26px;
  border-radius: 22px;
  margin-top: 28px;
  border: 4px solid #2d6a4f;
  box-shadow: 0 8px 22px rgba(0, 0, 0, 0.18);
  color: #081c15;
">
  <h1 style="color: #081c15; background-color: rgba(255,255,255,0.65); padding: 12px 16px; border-radius: 14px; margin-top: 0;">
    📡 Part 1 — Bayesian Reasoning with Simulation
  </h1>
  <p style="font-size: 17px; line-height: 1.7; color: #133f30; font-weight: 500;">
    Intuition and simulation of fundamental probability concepts: Law of Total Probability, Bayes' theorem,
    and how empirical frequencies converge to theoretical values as predicted by the Law of Large Numbers.
  </p>
</div>

<div style="
  background: linear-gradient(135deg, #a8edea, #fed6e3);
  padding: 24px;
  border-radius: 22px;
  margin-top: 28px;
  border: 4px solid #8e6fa0;
  box-shadow: 0 8px 20px rgba(0,0,0,0.12);
  color: #1a1a2e;
">
  <h1 style="color: #2b0a2a; background-color: rgba(255,255,255,0.55); padding: 10px 16px; border-radius: 14px; margin-top: 0; font-size: 26px;">
    ✈️ 1.1 Airport Security System — Law of Total Probability & Conditional Probability
  </h1>
  <p style="font-size: 16px; line-height: 1.7; color: #2b0a2a;">
    Scenario: an airport scanner detects prohibited objects. We derive <code>P(Alarm)</code> and
    <code>P(Object | Alarm)</code> and validate them with a Monte Carlo experiment.
  </p>
</div>

#### 1.1.1 Analytical Derivation

Given:

* $P(\text{Object}) = 0.05$
* $P(\text{Alarm} \mid \text{Object}) = 0.90$
* $P(\text{Alarm} \mid \neg\text{Object}) = 0.08$

**Law of Total Probability:**

$$P(\text{Alarm}) = P(\text{Alarm}\mid\text{Object})P(\text{Object}) + P(\text{Alarm}\mid\neg\text{Object})P(\neg\text{Object})$$

$$= 0.90 \times 0.05 + 0.08 \times 0.95 = 0.045 + 0.076 = 0.121$$

**Bayes' theorem:**

$$P(\text{Object}\mid\text{Alarm}) = \frac{P(\text{Alarm}\mid\text{Object})P(\text{Object})}{P(\text{Alarm})} = \frac{0.045}{0.121} = \frac{45}{121} \approx 0.3719$$

Note $P(\neg\text{Object}) = 1 - P(\text{Object}) = 0.95$ is used explicitly.


In [ ]:
# 1.1 — Analytical calculation (code implements the formulas)
p_object <- 0.05
p_alarm_given_object <- 0.90
p_alarm_given_no_object <- 0.08

p_alarm <- p_alarm_given_object * p_object + p_alarm_given_no_object * (1 - p_object)
p_object_given_alarm <- (p_alarm_given_object * p_object) / p_alarm

cat(sprintf("P(Alarm) = %.4f\n", p_alarm))
cat(sprintf("P(Object|Alarm) = %.4f  (45/121 = %.4f)\n", p_object_given_alarm, 45/121))


#### 1.1.2 Monte Carlo Simulation (Corrected)

**Fix vs original:** The original used `c(rep(1, N*p_object), rep(0, ...))` then `sample()`, which forces exactly $N\cdot p$ objects (zero variance, and breaks for non-integer $N\cdot p$ e.g. $N=10$). The correct Monte Carlo draws each passenger independently:

* `object_status ~ Bernoulli(p_object)`  via `rbinom(N, 1, p_object)`
* `alarm | object ~ Bernoulli(p_alarm_given_object)` and `alarm | \neg object ~ Bernoulli(p_alarm_given_no_object)` via `rbinom` per group.

This preserves $\text{Var}(\text{count}) = Np(1-p)$ and satisfies the Law of Large Numbers.


In [ ]:
# 1.1 — Simulation with correct Bernoulli draws (N = 100,000)
set.seed(42)
N <- 100000

# Each passenger independently has object with prob p_object
object_status <- rbinom(N, 1, p_object)

alarm <- integer(N)
idx_obj <- which(object_status == 1)
idx_no_obj <- which(object_status == 0)

alarm[idx_obj] <- rbinom(length(idx_obj), 1, p_alarm_given_object)
alarm[idx_no_obj] <- rbinom(length(idx_no_obj), 1, p_alarm_given_no_object)

emp_p_alarm <- mean(alarm == 1)
# Guard against zero alarms (NaN)
emp_p_object_given_alarm <- if (sum(alarm == 1) == 0) NA else mean(object_status[alarm == 1] == 1)

cat(sprintf("Empirical P(Alarm)         = %.4f (theoretical %.4f)\n", emp_p_alarm, p_alarm))
cat(sprintf("Empirical P(Object|Alarm)  = %.4f (theoretical %.4f)\n", emp_p_object_given_alarm, p_object_given_alarm))


#### 1.1.3 Analytical vs Empirical Comparison


In [ ]:
# 1.1 — Bar chart: analytical vs simulation
png("results/airport_comparison.png", width=900, height=700, res=120)
par(mar=c(5,5,4,2))
bar_positions <- barplot(
  height = rbind(c(p_alarm, p_object_given_alarm), c(emp_p_alarm, emp_p_object_given_alarm)),
  beside = TRUE,
  names.arg = c("P(Alarm)", "P(Object|Alarm)"),
  col = c("skyblue", "orange"),
  ylim = c(0, max(c(p_alarm, p_object_given_alarm, emp_p_alarm, emp_p_object_given_alarm), na.rm=TRUE) * 1.3),
  main = "Airport: Analytical vs Simulation (N = 100,000)",
  ylab = "Probability",
  border = "black",
  cex.names = 1.1, cex.lab = 1.1
)
legend("topright", legend = c("Analytical", "Simulation"), fill = c("skyblue", "orange"), bty = "n", cex=1.05)
grid(nx = NA, ny = NULL, col = "gray", lty = "dotted")
# annotate values
text(bar_positions, rbind(c(p_alarm, p_object_given_alarm), c(emp_p_alarm, emp_p_object_given_alarm)) + 0.015,
     labels = sprintf("%.3f", rbind(c(p_alarm, p_object_given_alarm), c(emp_p_alarm, emp_p_object_given_alarm))),
     cex=0.85)
dev.off()

bar_positions <- barplot(
  height = rbind(c(p_alarm, p_object_given_alarm), c(emp_p_alarm, emp_p_object_given_alarm)),
  beside = TRUE,
  names.arg = c("P(Alarm)", "P(Object|Alarm)"),
  col = c("skyblue", "orange"),
  ylim = c(0, max(c(p_alarm, p_object_given_alarm, emp_p_alarm, emp_p_object_given_alarm), na.rm=TRUE) * 1.3),
  main = "Airport: Analytical vs Simulation (N = 100,000)",
  ylab = "Probability",
  border = "black"
)
legend("topright", legend = c("Analytical", "Simulation"), fill = c("skyblue", "orange"), bty = "n")
grid(nx = NA, ny = NULL, col = "gray", lty = "dotted")


#### 1.1.4 Convergence Analysis

**Fix vs original:** Single run per $N$ has high variance (especially $N=10$ where $E[\text{alarm}]\approx1.2$ may be zero → `NaN`). The corrected version **replicates each $N$ $B=300$ times**, then plots the **mean ± SD** band. Convergence to the red theoretical line demonstrates the Law of Large Numbers.


In [ ]:
# 1.1 — Convergence: mean +- SD over B replicates per N
set.seed(42)
Ns <- c(10, 100, 500, 1000, 5000, 10000, 50000, 100000)
B <- 300

emp_means <- numeric(length(Ns))
emp_sds <- numeric(length(Ns))

for (i in seq_along(Ns)) {
  n <- Ns[i]
  reps <- replicate(B, {
    obj <- rbinom(n, 1, p_object)
    al <- integer(n)
    io <- which(obj==1); ino <- which(obj==0)
    if (length(io)>0) al[io] <- rbinom(length(io), 1, p_alarm_given_object)
    if (length(ino)>0) al[ino] <- rbinom(length(ino), 1, p_alarm_given_no_object)
    if (sum(al==1)==0) NA else mean(obj[al==1]==1)
  })
  emp_means[i] <- mean(reps, na.rm=TRUE)
  emp_sds[i] <- sd(reps, na.rm=TRUE)
}

# Table
print(data.frame(N=Ns, Mean=round(emp_means,4), SD=round(emp_sds,4), Theoretical=round(p_object_given_alarm,4)))

# Plot
png("results/airport_convergence.png", width=1000, height=600, res=120)
plot(Ns, emp_means, type="b", pch=16, col="steelblue", lwd=2, log="x",
     main = "Convergence of P(Object|Alarm) — Mean +- SD over 300 replicates",
     xlab = "Number of passengers (N, log scale)", ylab = "P(Object|Alarm)",
     ylim = c(0, max(c(emp_means+emp_sds, p_object_given_alarm), na.rm=TRUE)*1.25),
     xaxt="n")
axis(1, at=Ns, labels=Ns, cex.axis=0.85)
# SD band
polygon(c(Ns, rev(Ns)), c(emp_means-emp_sds, rev(emp_means+emp_sds)), col=rgb(0.27,0.51,0.71,0.25), border=NA)
lines(Ns, emp_means, type="b", pch=16, col="steelblue", lwd=2)
abline(h=p_object_given_alarm, col="red", lty=2, lwd=2)
points(Ns, emp_means, pch=16, col="steelblue", cex=1.2)
legend("bottomright", legend=c("Empirical mean +- SD", "Theoretical (45/121 = 0.372)"),
       col=c("steelblue","red"), lty=c(1,2), pch=c(16,NA), pt.cex=1.1, lwd=2, bty="n")
grid()
dev.off()

plot(Ns, emp_means, type="b", pch=16, col="steelblue", lwd=2, log="x",
     main = "Convergence of P(Object|Alarm) — Mean +- SD over 300 replicates",
     xlab = "Number of passengers (N, log scale)", ylab = "P(Object|Alarm)",
     ylim = c(0, max(c(emp_means+emp_sds, p_object_given_alarm), na.rm=TRUE)*1.25),
     xaxt="n")
axis(1, at=Ns, labels=Ns, cex.axis=0.85)
polygon(c(Ns, rev(Ns)), c(emp_means-emp_sds, rev(emp_means+emp_sds)), col=rgb(0.27,0.51,0.71,0.25), border=NA)
lines(Ns, emp_means, type="b", pch=16, col="steelblue", lwd=2)
abline(h=p_object_given_alarm, col="red", lty=2, lwd=2)
points(Ns, emp_means, pch=16, col="steelblue", cex=1.2)
legend("bottomright", legend=c("Empirical mean +- SD", "Theoretical (45/121 = 0.372)"),
       col=c("steelblue","red"), lty=c(1,2), pch=c(16,NA), pt.cex=1.1, lwd=2, bty="n")
grid()


<div style="
  background: linear-gradient(135deg, #e8b7e8, #ffb6f4);
  padding: 24px;
  border-radius: 22px;
  margin-top: 28px;
  border: 4px solid #481344;
  box-shadow: 0 8px 20px rgba(0,0,0,0.12);
  color: #2b0a2a;
">
  <h1 style="color: #2b0a2a; background-color: rgba(255,255,255,0.55); padding: 10px 16px; border-radius: 14px; margin-top: 0; font-size: 26px;">
    💳 1.2 Fraud Detection — Bayes' Theorem with One and Two Features
  </h1>
  <p style="font-size: 16px; line-height: 1.7; color: #2b0a2a;">
    Single-feature posterior <code>P(Fraud | OddCountry)</code> and the Naive Bayes extension
    <code>P(Fraud | OddCountry, LargeAmount)</code> under conditional independence.
  </p>
</div>

#### 1.2.1 Analytical Derivation — Single Feature

Given:

* $P(\text{Fraud}) = 0.02$
* $P(\text{OddCountry} \mid \text{Fraud}) = 0.70$
* $P(\text{OddCountry} \mid \neg\text{Fraud}) = 0.05$
* $P(\text{LargeAmount} \mid \text{Fraud}) = 0.60$
* $P(\text{LargeAmount} \mid \neg\text{Fraud}) = 0.10$

**Step 1 — $P(\text{OddCountry})$ via LTP:**

$$P(\text{Odd}) = 0.70\times0.02 + 0.05\times0.98 = 0.014 + 0.049 = 0.063$$

**Step 2 — Bayes:**

$$P(\text{Fraud}\mid\text{Odd}) = \frac{0.70\times0.02}{0.063} = \frac{0.014}{0.063} = \frac{14}{63} \approx 0.2222$$

**Step 3 — Two features (Naive Bayes assumption):**

Assume **conditional independence** given fraud:

$$P(\text{Odd},\text{Large}\mid\text{Fraud}) = P(\text{Odd}\mid\text{Fraud})\cdot P(\text{Large}\mid\text{Fraud}) = 0.42$$

$$P(\text{Odd},\text{Large}\mid\neg\text{Fraud}) = 0.05\times0.10 = 0.005$$

Then

$$P(\text{Odd},\text{Large}) = 0.42\times0.02 + 0.005\times0.98 = 0.0084 + 0.0049 = 0.0133$$

$$P(\text{Fraud}\mid\text{Odd},\text{Large}) = \frac{0.0084}{0.0133} = \frac{84}{133} \approx 0.6316$$

> **Fix vs original:** Comment `84/113` corrected to `84/133`; added explicit conditional-independence statement and corrected "OldCountry" typo.


In [ ]:
# 1.2 — Analytical
p_fraud <- 0.02
p_odd_if_fraud <- 0.70
p_odd_if_no <- 0.05
p_large_if_fraud <- 0.60
p_large_if_no <- 0.10

p_odd <- p_odd_if_fraud * p_fraud + p_odd_if_no * (1 - p_fraud)
p_fraud_if_odd <- (p_odd_if_fraud * p_fraud) / p_odd

p_joint <- p_odd_if_fraud * p_large_if_fraud * p_fraud + p_odd_if_no * p_large_if_no * (1 - p_fraud)
p_fraud_if_odd_large <- (p_odd_if_fraud * p_large_if_fraud * p_fraud) / p_joint

cat(sprintf("P(Odd) = %.4f\n", p_odd))
cat(sprintf("P(Fraud|Odd) = %.4f  (14/63 = %.4f)\n", p_fraud_if_odd, 14/63))
cat(sprintf("P(Odd,Large) = %.4f\n", p_joint))
cat(sprintf("P(Fraud|Odd,Large) = %.4f  (84/133 = %.4f)\n", p_fraud_if_odd_large, 84/133))
